# 1. Demand shocks and empirical workload

This notebook reconstructs the empirical workload used by **Dynamic Simulation of a Bundle-Priced EIP-7999 Multidimensional Fee Market**.

It produces the three price-adjusted parent-demand shocks—execution, static data, and state creation—and the BAL access-composition shock. The source factors are normalized at the distribution level. Individual simulated paths are not recentered, so sampled weeks can remain unusually busy or quiet.

The simulation is conditional on metering, BAL-decomposition, and elasticity handoffs produced by the two preceding publication workflows:

1. notebooks/resource_demand_and_glamsterdam_equilibrium/01–03
2. notebooks/7999_equilibrium/01–02

Those notebooks contain the Xatu and RPC procedures used to construct the historical accounting and model anchors. This notebook adds the contiguous Xatu/RPC pulls needed to measure block-to-block persistence.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import Image, display

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src").is_dir() and (candidate / "scripts").is_dir():
            return candidate
    raise RuntimeError("Could not locate the repository root")

PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
for path in (SRC_DIR, SCRIPTS_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

DATA_DIR = PROJECT_ROOT / "data"
OUT_DIR = DATA_DIR / "7999"
PLOTS_DIR = PROJECT_ROOT / "plots"
OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

REFRESH_FROM_NETWORK = (
    os.environ.get("REFRESH_7999_SIMULATION_FROM_NETWORK", "0") == "1"
)
REUSE_OUTPUTS = os.environ.get("REUSE_7999_SIMULATION_OUTPUTS", "0") == "1"

def run_script(script: str, *args: str) -> None:
    command = [sys.executable, str(SCRIPTS_DIR / script), *map(str, args)]
    print("+", " ".join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

print("project root:", PROJECT_ROOT)
print("network refresh:", REFRESH_FROM_NETWORK)
print("reuse generated outputs:", REUSE_OUTPUTS)

## Network refresh

Set REFRESH_7999_SIMULATION_FROM_NETWORK=1 before execution to rebuild the ignored contiguous panels.

The refresh reads CLICKHOUSE_USER and CLICKHOUSE_PASSWORD from .env. Optional ClickHouse endpoint overrides are CLICKHOUSE_RAW_HOST and CLICKHOUSE_PORT. The runtime-BAL pull queries Xatu execution events, traces, and structlogs through the repository's EIP-8279 meter; it is chunked and resumes from partial CSV files.

The primitive panel is pulled from Xatu for 60 consecutive days. Runtime BAL is reconstructed over the same block range in two adjacent pieces so the expensive query can be resumed independently. RPC-derived metering and BAL-calibration inputs are produced by the upstream publication notebooks using ETHNODEOPS_API_KEY or ALCHEMY_RPC.

In [ ]:
UPSTREAM_INPUTS = [
    DATA_DIR / "daily_accounting_panel_calibrated_with_bal_2026-02-01_2026-06-01.csv",
    DATA_DIR / "daily_current_data_gas_xatu_2026-02-01_2026-06-01.csv",
    DATA_DIR / "calibration_state_access_auth_daily_rates_2026-02-01_2026-06-01.csv",
    DATA_DIR / "glamsterdam/elasticity_vectors.csv",
    OUT_DIR / "data_metering_runtime_bal_anchor.csv",
    OUT_DIR / "bal_decomposition_demand_parameters.csv",
]
missing_upstream = [path for path in UPSTREAM_INPUTS if not path.exists()]
if missing_upstream:
    missing = "\n".join(f"  - {path.relative_to(PROJECT_ROOT)}" for path in missing_upstream)
    raise FileNotFoundError(
        "Missing upstream publication handoffs:\n"
        f"{missing}\n"
        "Run the numbered notebooks in resource_demand_and_glamsterdam_equilibrium "
        "and 7999_equilibrium first."
    )

if REFRESH_FROM_NETWORK:
    run_script(
        "build_contiguous_block_panel.py",
        "--start", "2026-04-02",
        "--days", "60",
        "--out-dir", "data/contiguous",
    )
    run_script(
        "build_contiguous_runtime_bal.py",
        "--range", "24788193", "25118358",
        "--label", "hist60d",
        "--chunk-size", "250",
    )
    run_script(
        "build_contiguous_runtime_bal.py",
        "--range", "25118359", "25218797",
        "--label", "full14d",
        "--chunk-size", "250",
    )
else:
    print("Using cached contiguous Xatu/runtime-BAL panels.")

## Build the canonical multiscale paths

The fast residual vector is resampled jointly in 3,200-block strips. Daily factors are sampled jointly in contiguous eight-day strips, and the recurring UTC-hour profile is restored before simulation. All later notebooks use the same seeds and constructors.

In [ ]:
import build_multiscale_demand_shocks

build_multiscale_demand_shocks.main()

## Validate the workload handoff

The manifest fixes the source range, path count, burn-in, measurement window, bootstrap lengths, and random seeds. The per-path means demonstrate that paths were not normalized individually.

In [ ]:
manifest = pd.read_csv(OUT_DIR / "multiscale_workload_manifest.csv")
shock_summary = pd.read_csv(OUT_DIR / "multiscale_workload_shock_summary.csv")
path_means = pd.read_csv(
    OUT_DIR / "multiscale_path_mean_factors.csv", index_col="replication"
)
round_trip = pd.read_csv(OUT_DIR / "multiscale_source_round_trip.csv")

display(manifest)
display(shock_summary)
display(path_means.describe().T[["min", "mean", "max"]])
display(round_trip)

assert manifest.loc[0, "fast_source_start_block"] == 24_788_193
assert manifest.loc[0, "fast_source_end_block"] == 25_218_797
assert manifest.loc[0, "paths"] == 32
assert manifest.loc[0, "measured_blocks"] == 50_400
assert not bool(manifest.loc[0, "path_specific_normalization"])
assert round_trip["max_abs_log_reconstruction_error"].max() <= 1e-12
assert not np.allclose(path_means.to_numpy(), 1.0, rtol=0.0, atol=1e-12)

## Handoff

This notebook writes the workload manifest, hourly profile, daily factors, source positions, path-level shock summaries, and round-trip diagnostics under data/7999/. Notebook 2 uses the same deterministic workload construction to run the fee-market target grid.